In [ ]:
import os
from getpass import getpass


os.environ["GOOGLE_API_KEY"] = getpass("Enter your Gemini API Key: ")

In [ ]:
!pip -q install \
langchain \
langchain-community \
langchain-google-genai \
faiss-cpu \
pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.7/70.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 23.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip -q install -U langchain langchain-community langchain-text-splitters

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 136.9/136.9 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.3/558.3 kB 18.9 MB/s eta 0:00:00


In [ ]:
!pip -q install sentence-transformers

In [ ]:
!pip -q install langchain-huggingface

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

pdf_files = [

    "rag_building1.pdf",
    "rag_building2.pdf",
    "rag_building3.pdf",
    "rag_building4.pdf"
]

documents = []
for pdf in pdf_files:
    loader = PyPDFLoader(pdf)
    docs = loader.load()
    documents.extend(docs)

print(f"Total Pages Loaded: {len(documents)}")

Total Pages Loaded: 69


In [ ]:
# -------------------------------
# 2. Split into Chunks
# -------------------------------
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

chunks = text_splitter.split_documents(documents)
print(f"Total Chunks: {len(chunks)}")

Total Chunks: 319


In [ ]:

enriched_chunks = []
for i, chunk in enumerate(chunks):
    enriched_chunks.append(
        Document(
            page_content=chunk.page_content,
            metadata={
                "source": chunk.metadata.get("source", ""),
                "page": chunk.metadata.get("page", 0),
                "chunk_id": f"chunk_{i}",
                "keywords":["Attention", "LLM", "Rag Pipeline","Transformers"],  # customize per paper
                "document_type": "research_paper"
            }
        )
    )


In [ ]:
# -------------------------------
# 4. Create Vector DB
# -------------------------------
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector_db = FAISS.from_documents(
    enriched_chunks,
    embeddings
)

print("Vector database created!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Vector database created!


In [ ]:
# -------------------------------
# 5. Load Gemini LLM
# -------------------------------
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    temperature=0
)

print("Gemini Loaded!")

Gemini Loaded!


In [ ]:
# -------------------------------
# 6. Retriever with Metadata Filter
# -------------------------------
retriever = vector_db.as_retriever(
    search_kwargs={
        "k": 8,
        "filter": {"document_type": "research_paper"}  # filter ensures only research papers
    }
)

print("Retriever Ready!")

Retriever Ready!


In [ ]:
# -------------------------------
# 7. Ask RAG Function
# -------------------------------
def ask_rag(question):
    # Retrieve relevant chunks
    docs = retriever.invoke(question)

    # Combine retrieved text
    context = "\n\n".join(doc.page_content for doc in docs)

    # Prompt for Gemini
    prompt = f"""
You are a helpful AI assistant.

Answer ONLY using the information in the provided context.

If the answer is not available in the context, reply:

"I couldn't find that information in the uploaded PDFs."

Context:
{context}

Question:
{question}

Answer:
"""

    # Generate response
    response = llm.invoke(prompt)

    print("=" * 80)
    print("QUESTION:")
    print(question)

    print("\nANSWER:")
    print(response.content)

    print("\nSOURCES:")
    for doc in docs:
        meta = doc.metadata
        print(
            f" {meta['source']} | Page {meta['page'] + 1} | "
            f"Chunk: {meta.get('chunk_id','N/A')} | "
            f"Keywords: {', '.join(meta.get('keywords', []))}"
        )

    print("=" * 80)

In [ ]:
# -------------------------------
# 8. Interactive Loop
# -------------------------------
while True:
    question = input("\nAsk a question (type 'exit' to stop): ")
    if question.lower() == "exit":
        break
    ask_rag(question)


Ask a question (type 'exit' to stop): what is RAG?
QUESTION:
what is RAG?

ANSWER:
RAG is a model that can be employed as a language model. It is strongly grounded in real factual knowledge, such as Wikipedia, which helps it to "hallucinate" less and produce more factual generations, offering more control and interpretability. RAG can be used for open-domain questions and sequence classification tasks.

SOURCES:
 rag_building2.pdf | Page 17 | Chunk: chunk_134 | Keywords: Attention, LLM, Rag Pipeline, Transformers
 rag_building3.pdf | Page 17 | Chunk: chunk_226 | Keywords: Attention, LLM, Rag Pipeline, Transformers
 rag_building2.pdf | Page 10 | Chunk: chunk_99 | Keywords: Attention, LLM, Rag Pipeline, Transformers
 rag_building3.pdf | Page 10 | Chunk: chunk_191 | Keywords: Attention, LLM, Rag Pipeline, Transformers
 rag_building2.pdf | Page 8 | Chunk: chunk_92 | Keywords: Attention, LLM, Rag Pipeline, Transformers
 rag_building3.pdf | Page 8 | Chunk: chunk_184 | Keywords: Attention, L